# Step 9 v2 — MiniImageNet through the GPU relay

This notebook preserves Step 9's ten 5-way 5-shot experiments while keeping control, manifests, checkpoints, consolidation, and final artifacts on this machine.
Only CUDA-dependent initialization, epoch partitions, temperature fitting, evaluation shards, and checkpoint export run on the selected Kaggle T4.
Every completed boundary is hash-checked and restartable; the optional 1-shot extension remains disabled by default.

## 1. Required local environment and data

Run this notebook from the thesis repository after installing `requirements-vgpu.txt` in the local kernel environment.
Set `GPU_POOL_URL` and `GPU_POOL_API_KEY` in `.env`; datasets are discovered automatically from this repository's populated `data/` folder.
`VGPU_DRIVE_DIR` is optional and must name an already-mounted directory; local state remains authoritative.

In [1]:
import atexit, json, os, subprocess, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'plan.txt').is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'plan.txt').is_file(), 'start inside the thesis repository'
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.vgpu import (
    VGPU_RUNS, vgpu_bootstrap_cell, vgpu_build_data_manifest,
    vgpu_build_source_bundle, vgpu_close, vgpu_create_controller,
    vgpu_finalize_manifest, vgpu_resume_run, vgpu_run_integration_gate,
    vgpu_run_step9, vgpu_stage_run, vgpu_step9_consolidate,
    vgpu_resolve_data_dir, vgpu_verify_client, vgpu_worker_sha256,
)
from src.vgpu.controller import vgpu_load_env

RUN_ID = os.environ.get('VGPU_RUN_ID', 'step9_v2_main')
env = vgpu_load_env(REPO_ROOT / '.env')
required = ['GPU_POOL_URL', 'GPU_POOL_API_KEY']
missing = [key for key in required if not env.get(key)]
assert not missing, f'missing .env values: {missing}'
print('repo:', REPO_ROOT)
print('run :', RUN_ID)
print('data: repository data/ (automatic)')


repo: /home/fat-potato/prj/thesis
run : step9_v2_main
data: repository data/ (automatic)


## 2. Temporary Kaggle worker bootstrap

On a fresh Kaggle relay notebook, run the normal sharedGPU CONFIG cell first.
Copy the generated cell below into Kaggle and run it before sharedGPU's JOIN cell; it writes one checksum-pinned, untracked `pool_models/vgpu_step9.py`.
The bootstrap does not commit, reset, or otherwise edit sharedGPU history.

In [2]:
print('expected worker SHA-256:', vgpu_worker_sha256(REPO_ROOT))
print('\nCOPY THE CELL BELOW TO KAGGLE (after CONFIG, before JOIN):\n')
print(vgpu_bootstrap_cell(REPO_ROOT))


expected worker SHA-256: 9f6b0082625f3af5d5cdf03913f7111d09efd83aaee5d1df72025f688881e053

COPY THE CELL BELOW TO KAGGLE (after CONFIG, before JOIN):

# Step 9 v2 temporary worker bootstrap (run after CONFIG, before JOIN)
import base64, hashlib, os, subprocess, zlib
from pathlib import Path
repo = os.environ.get("GPU_POOL_REPO") or globals().get("GPU_POOL_REPO")
if not repo:
    raise RuntimeError("run sharedGPU's CONFIG cell so GPU_POOL_REPO is set")
clone = Path("/kaggle/working/gpu_pool_repo")
if not (clone / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", repo, str(clone)], check=True)
target = clone / "pool_models" / "vgpu_step9.py"
raw = zlib.decompress(base64.b64decode("eNrdHGtz47bxu34Fwi8hHR3Pd3m0UaPOXFPn0Usu6d0laep6eBQJSYwpkuXDtuLqv3d3AZB4UJac+NJMPXMnClgAi8W+sZTnea/auEjjvCw4a/mmKuu43rLn8WqVc3Zd1pe8ZsuyZq9aXrGP2dXTcDJ5vc4atswAAD6Tssp4yk66oq3j5JKnJ6wt2Zs3VVnm0aZMed48vlpVXdTADB+H1fbNG7bYsnbNJ4uybBsYVbGE53nI2JctA/hsweu45fmWreOGFSXLEK0WlqzLDQ7Efw0sXcF68YozWDrL

## 3. Local compilation, configs, and tests

This section compiles the additive vGPU package and then runs the complete repository test suite locally.
It also hashes the source bundle and every dataset file before any relay connection is made.
A failure here blocks GPU work; install the repository requirements and `pytest` in this notebook's local kernel if imports are missing.

In [3]:
vgpu_verify_client(REPO_ROOT)
compile_targets = sorted(
    list((REPO_ROOT / 'src').rglob('*.py'))
    + list((REPO_ROOT / 'scripts').glob('*.py'))
    + list((REPO_ROOT / 'tests').glob('*.py'))
)
subprocess.run(
    [sys.executable, '-m', 'py_compile', *map(str, compile_targets)],
    cwd=REPO_ROOT, check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=REPO_ROOT, check=True,
)
print(f'compiled {len(compile_targets)} repository modules; full pytest passed')


........................................................................ [ 33%]
........................................................................ [ 67%]
.....................................................................    [100%]
213 passed in 26.21s
compiled 95 repository modules; full pytest passed


In [4]:
preflight_root = REPO_ROOT / 'data' / 'vgpu_step9_state' / RUN_ID
preflight_root.mkdir(parents=True, exist_ok=True)
source_info = vgpu_build_source_bundle(
    REPO_ROOT, preflight_root / 'source.tar.gz'
)
data_info = vgpu_build_data_manifest(vgpu_resolve_data_dir(REPO_ROOT))
print(json.dumps({
    'worker_sha256': vgpu_worker_sha256(REPO_ROOT),
    'source_sha256': source_info['sha256'],
    'source_files': source_info['file_count'],
    'data_manifest_sha256': data_info['sha256'],
    'data_files': data_info['file_count'],
    'data_gib': round(data_info['total_bytes'] / 1024**3, 3),
}, indent=2))


{
  "worker_sha256": "9f6b0082625f3af5d5cdf03913f7111d09efd83aaee5d1df72025f688881e053",
  "source_sha256": "d186223cce35c840c961067f110474beab9d234f1a9e2cb02611a0994ea5672b",
  "source_files": 105,
  "data_manifest_sha256": "f604ce93368411ab5d5090bc4c4a63d7d75f215c72011b45e12fbbd8f9b75d61",
  "data_files": 120214,
  "data_gib": 2.356
}


## 4. Verified relay probe and pinned worker

The exact probe from `docs/gpu-relay-guide.md` must finish with its literal `VERIFIED` line.
After that teardown, the controller selects a non-draining Kaggle T4 with at least 12 GiB scheduler-reported free VRAM and four hours of quota.
The deployment is accepted only when Kaggle reports the same temporary-worker checksum printed locally.

In [5]:
probe = subprocess.run(
    [sys.executable, '-u', 'docs/verified_gpu_pool_probe.py'],
    cwd=REPO_ROOT, text=True, capture_output=True,
)
print(probe.stdout)
if probe.stderr:
    print(probe.stderr, file=sys.stderr)
probe.check_returncode()
assert probe.stdout.rstrip().endswith(
    'VERIFIED: vendored client -> control plane -> remote CUDA -> teardown'
)


{
  "clientSha256": "a68de6e03e639b77aabcccf55b36719e2702f1b9fee04cee8393302a65345fee",
  "inferTimeoutSeconds": 120.0,
  "loadTimeoutSeconds": 600.0,
  "poolNodeCount": 1,
  "selectedNode": {
    "freeVramGib": 14.56,
    "gpuName": "Tesla T4",
    "nodeId": "node-1e800f602021",
    "provider": "kaggle",
    "quotaRemainingSeconds": 28687.843957662582
  },
  "verifiedClient": "/home/fat-potato/prj/thesis/mainul-doc/client.py"
}
{
  "probe": {
    "checksum": 477375.0625,
    "computeCapability": "7.5",
    "cuda": true,
    "device": "Tesla T4",
    "host": "a790332264dd",
    "matmulMs": 7.09,
    "matmulSize": 4096,
    "roundTripSeconds": 0.537,
    "vramFreeGib": 14.28,
    "vramTotalGib": 14.56
  }
}
Teardown confirmed for dep-382302a46dce
VERIFIED: vendored client -> control plane -> remote CUDA -> teardown



In [6]:
controller = None
controller, preparation = vgpu_create_controller(
    REPO_ROOT, run_id=RUN_ID,
    drive_dir=env.get('VGPU_DRIVE_DIR'),
)
atexit.register(lambda: vgpu_close(controller) if controller is not None else None)
resume = vgpu_resume_run(controller)
print('node:', preparation['node'])
print('worker:', preparation['remote_status'])
print('resume state:', resume['state'])


node: {'nodeId': 'node-1e800f602021', 'provider': 'kaggle', 'accountLabel': 'Mainul-kaggle', 'gpuName': 'Tesla T4', 'offeredVramGib': 14.56, 'totalVramGib': 14.56, 'freeVramGib': 14.56, 'reservedVramGib': 0.0, 'deployments': [], 'deploymentCount': 0, 'dirtyDeployments': [], 'draining': False, 'quotaRemainingSeconds': 28657.83118057251, 'connectedAtUnix': 1785362725.003, 'uptimeSeconds': 147.6, 'lastHeartbeatAgeSeconds': 7.2}
worker: {'worker_version': 1, 'worker_sha256': '9f6b0082625f3af5d5cdf03913f7111d09efd83aaee5d1df72025f688881e053', 'cuda': True, 'device': 'Tesla T4', 'vram_free_gib': 14.22, 'vram_total_gib': 14.56, 'root': '/kaggle/working/vgpu_step9', 'runs': []}
resume state: idle


## 5. Resumable source and data staging

The first pass deliberately stops one source chunk and one data-file chunk, then resumes from the worker-reported exact offsets.
All later chunks use 16 MiB raw payloads with per-chunk and whole-file SHA-256 checks.
On restart, already-complete remote files are verified and skipped; extraction is restricted to this run's Kaggle workspace.

In [7]:
try:
    staged = vgpu_stage_run(controller, preparation)
    print(json.dumps(staged, indent=2, sort_keys=True))
except BaseException:
    vgpu_close(controller)
    controller = None
    raise


integration gate: interrupt source upload after one chunk
data 1/120214: cifar-100-python/file.txt~ (0.0 MB)
integration gate: interrupt first data upload after one chunk
data 2/120214: cifar-100-python/meta (0.0 MB)
data 3/120214: cifar-100-python/test (31.0 MB)
data 4/120214: cifar-100-python/train (155.2 MB)
data 5/120214: cifar_fs_split.json (0.0 MB)
data 6/120214: mini-imagenet-cache-test.pkl (353.6 MB)
data 7/120214: mini-imagenet-cache-train.pkl (1145.5 MB)


KeyboardInterrupt: 

## 6. Remote dataset/model and recovery gate

The first ResNet-18 post-pool evidential initialization loads the real MiniImageNet train/val/test splits and constructs the configured CUDA model.
Epoch 1 is committed locally, restored through a fresh initialization, and followed by epoch 2 before the configuration continues.
The gate validates two ordered 50-episode shards and a strict reload of the exported full checkpoint; it completes the first configuration before unlocking the matrix.

In [ ]:
try:
    gate = vgpu_run_integration_gate(controller)
    print('integration gate:', gate.get('config_name'), 'passed')
except BaseException:
    vgpu_close(controller)
    controller = None
    raise


## 7. Ten-configuration partitioned Step 9 run

The fixed order is ResNet-18 post-pool, ResNet-18 parallel, MobileNetV3 post-pool, MobileNetV3 parallel, then ResNet-18 linear-probe; each pair runs evidential before softmax.
Training is one submitted job per epoch and evaluation is twelve submitted shards of 50 frozen test episodes.
Verified completed partitions are reported and skipped, so this same cell is the normal restart entry point after reconnecting and staging.

In [ ]:
try:
    five_shot = vgpu_run_step9(controller, k_shot=5)
    print('5-shot configurations:', len(five_shot))
except BaseException:
    vgpu_close(controller)
    controller = None
    raise


## 8. Optional 1-shot extension (disabled)

Step 9 v2 is complete with the ten 5-shot configurations; this block is not part of its acceptance gate.
Set `RUN_ONESHOT = True` only when the additional compute and storage are intentional.
The 1-shot checkpoints, shards, and metrics use distinct names and cannot mix with the mandatory 5-shot state.

In [ ]:
RUN_ONESHOT = False
if RUN_ONESHOT:
    try:
        one_shot = vgpu_run_step9(controller, k_shot=1)
        print('1-shot configurations:', len(one_shot))
    except BaseException:
        vgpu_close(controller)
        controller = None
        raise
else:
    print('1-shot extension disabled')


## 9. Local consolidation and artifact manifest

The unchanged Step 9 consolidation script runs locally through the additive wrapper and reads the ten canonical MiniImageNet metrics plus existing CIFAR-FS results.
It produces the dataset table and the two Step 9 comparison plots without running GPU code.
The final manifest hashes canonical results, full checkpoints, compact recovery checkpoints, and evaluation shards.

In [ ]:
try:
    vgpu_step9_consolidate(REPO_ROOT)
    manifest_path = vgpu_finalize_manifest(controller)
    print('manifest:', manifest_path)
    print(json.dumps(json.loads(
        (REPO_ROOT / 'results' / 'phase5_dataset_table.json').read_text()
    ), indent=2, sort_keys=True))
except BaseException:
    vgpu_close(controller)
    controller = None
    raise


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(REPO_ROOT / 'results' / 'step9_dataset_comparison.png')))
display(Image(filename=str(REPO_ROOT / 'results' / 'step9_adapter_uplift.png')))


## 10. Recovery summary, teardown, and thesis transcription

This final check requires ten completed 5-shot manifest cells, ten metrics files with exactly 600 episodes, and ten full checkpoints with matching hashes.
Confirmed teardown runs whether the acceptance assertions pass or fail; a teardown error remains recorded locally instead of being hidden.
Transcribe only values from `phase5_dataset_table.json` and the ten canonical metrics files into `step_writeups/step9.txt`, and record any collapse or relay failure from the run's `logs/` directory.

In [ ]:
try:
    manifest = json.loads(controller.state.manifest_path.read_text())
    completed = []
    for run in VGPU_RUNS:
        key = f"{run['config']}:5shot"
        cell = manifest.get('configs', {}).get(key, {})
        assert cell.get('evaluation_complete'), f'incomplete: {key}'
        metrics = Path(cell['metrics'])
        checkpoint = Path(cell['full_checkpoint'])
        assert metrics.is_file() and checkpoint.is_file()
        payload = json.loads(metrics.read_text())
        assert payload['num_episodes'] == 600
        completed.append(key)
    assert len(completed) == 10
    print('verified complete configurations:', len(completed))
    print('state:', controller.state.root)
    print('next: transcribe results/phase5_dataset_table.json into step_writeups/step9.txt')
finally:
    if controller is not None:
        vgpu_close(controller)
        controller = None
        print('relay deployment teardown confirmed')
